# Doubt Clearing Session: LLaMA Context Window + Ollama
This session answers:
- Q1: Context window issues with LLaMA and how to handle them
- Q2: What context window means, with chunking and streaming demos
- Q3: CPU vs GPU usage in Ollama and required configuration

> Local setup: `ollama` with model `llama3:latest`

#### Bala Satish

In [ ]:
# Setup for all demos (student-friendly version)
import json
import textwrap
import urllib.request
import urllib.error

# Keep this False in class so the notebook runs without Ollama installed.
USE_REAL_OLLAMA = False
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "llama3:latest"

def simple_local_answer(prompt):
    """Tiny local fallback so students can run everything without external tools."""
    p = prompt.lower()
    if "context window" in p:
        return "Context window means how much text the model can read in one request."
    if "summarize" in p:
        return "Summary: break big text into smaller parts, then combine the results."
    if "sla" in p or "p1" in p:
        return "From the provided context: P1 incidents require a 15-minute initial response."
    return "Demo answer: use clear prompts, relevant context, and small chunks."

def _stream_text(text):
    """Yield small pieces so streaming looks like token-by-token output."""
    for word in text.split():
        yield word + " "

def ollama_generate(prompt, model=MODEL, stream=False, options=None):
    """Return a full answer (stream=False) or a stream iterator (stream=True)."""
    # Classroom default: local fallback keeps notebook simple and reliable.
    if not USE_REAL_OLLAMA:
        answer = simple_local_answer(prompt)
        return _stream_text(answer) if stream else answer

    # Optional real API path for advanced students.
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": stream,
        "options": options or {}
    }
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(
        OLLAMA_URL,
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            if not stream:
                body = resp.read().decode("utf-8")
                return json.loads(body).get("response", "")

            def _real_stream():
                for raw_line in resp:
                    line = raw_line.decode("utf-8").strip()
                    if not line:
                        continue
                    obj = json.loads(line)
                    piece = obj.get("response", "")
                    if piece:
                        yield piece
                    if obj.get("done") is True:
                        break
            return _real_stream()
    except urllib.error.URLError as e:
        raise RuntimeError(
            "Could not reach Ollama. Keep USE_REAL_OLLAMA=False for class demos."
        ) from e

print("Setup loaded.")
print(f"Using model label: {MODEL}")
print(f"Real Ollama calls enabled: {USE_REAL_OLLAMA}")

Setup loaded.
Using model label: llama3:latest
Real Ollama calls enabled: False


## Q1. I am facing context window size issues in LLaMA. How do I handle this?
Use a token budget strategy:
1. Reserve tokens for output
2. Chunk long input with overlap
3. Send only the most relevant chunks
4. Summarize older context instead of sending everything

> In Ollama, `num_ctx` controls context size per request.

In [ ]:
# Q1 Demo: Context budget + chunking helper

# Simple numbers for classroom understanding.
MAX_CONTEXT = 4096
RESERVED_FOR_OUTPUT = 512

def chunk_text(text, chunk_chars=1200, overlap=200):
    """Split text into overlapping character chunks."""
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_chars, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = end - overlap

    return chunks

def build_budgeted_prompt(user_query, long_text):
    # 1) Make chunks
    chunks = chunk_text(long_text, chunk_chars=1200, overlap=200)

    # 2) Keep only first 3 chunks for a small demo prompt
    selected_chunks = chunks[:3]

    # 3) Build the final prompt
    prompt = (
        "You are a helpful assistant. Use the context to answer clearly.\n\n"
        f"Question: {user_query}\n\n"
        "Context:\n" + "\n\n---\n\n".join(selected_chunks)
    )

    # 4) Return prompt + quick stats
    return prompt, {
        "max_context": MAX_CONTEXT,
        "reserved_for_output": RESERVED_FOR_OUTPUT,
        "available_for_input": MAX_CONTEXT - RESERVED_FOR_OUTPUT,
        "chunks_total": len(chunks),
        "chunks_used": len(selected_chunks),
    }

sample_text = "LLaMA context management. " * 300
prompt, stats = build_budgeted_prompt(
    "What are best practices to handle context window limits?",
    sample_text
)

print("Q1 Stats:", stats)
print("Prompt preview:\n", prompt[:500], "...")

Q1 Stats: {'max_context': 4096, 'reserved_for_output': 512, 'available_for_input': 3584, 'chunks_total': 8, 'chunks_used': 3}
Prompt preview:
 You are a helpful assistant. Use the context to answer clearly.

Question: What are best practices to handle context window limits?

Context:
LLaMA context management. LLaMA context management. LLaMA context management. LLaMA context management. LLaMA context management. LLaMA context management. LLaMA context management. LLaMA context management. LLaMA context management. LLaMA context management. LLaMA context management. LLaMA context management. LLaMA context management. LLaMA context manage ...


## Q2. What is context window? Show chunking or streaming example
Context window is the maximum number of tokens the model can process in one request.
You must keep:
`input_tokens + output_tokens <= context_window`

> The next cell shows both:
- Chunking + map-reduce style summarization
- Streaming response from Ollama

In [ ]:
# Q2 Demo A: Chunking + simple map-reduce summarization

def summarize_chunk(chunk_text_value):
    """Simple summary rule for class: use the first sentence-like part."""
    sentence = chunk_text_value.strip().split(".")[0].strip()
    if not sentence:
        return "Short summary unavailable."
    return sentence + "."

def summarize_long_text_with_chunks(text):
    chunks = chunk_text(text, chunk_chars=500, overlap=80)
    partial_summaries = []

    # Map step: summarize each chunk
    for i, ch in enumerate(chunks, start=1):
        one_line = summarize_chunk(ch)
        partial_summaries.append(f"Chunk {i}: {one_line}")

    # Reduce step: combine into one final summary
    final_summary = " ".join(partial_summaries[:4])
    return final_summary

long_doc = textwrap.dedent("""
LLaMA has a fixed context window. If input is too long, some parts can be missed.
Chunking helps split long input into manageable pieces.
Overlap helps preserve continuity between neighboring chunks.
Streaming improves user experience by showing output as it is generated.
""") * 10

print("Running chunked summarization...")
result = summarize_long_text_with_chunks(long_doc)
print("\nFinal summary:\n", result)

print("\n" + "=" * 80)
print("Q2 Demo B: Streaming output")
stream_prompt = "Explain context window to students with one practical example."

for piece in ollama_generate(
    prompt=stream_prompt,
    options={"num_ctx": 1024, "temperature": 0.2},
    stream=True
):
    print(piece, end="", flush=True)
print()

Running chunked summarization...

Final summary:
 Chunk 1: LLaMA has a fixed context window. Chunk 2: helps preserve continuity between neighboring chunks. Chunk 3: context window. Chunk 4: ntinuity between neighboring chunks.

Q2 Demo B: Streaming output
Context window means how much text the model can read in one request. 


## Q3. When does LLaMA use CPU vs GPU in Ollama? Do we need config?
- Ollama auto-uses GPU when available and compatible.
- If no supported GPU is found, it runs on CPU.
- You can still tune behavior via options like `num_gpu`, `num_thread`, and `num_ctx`.

> The next cell demonstrates CPU-only and GPU-preferred API calls.

In [ ]:
# Q3 Demo: CPU vs GPU concept using easy-to-read options

def demo_cpu_gpu_calls():
    test_prompt = "Answer in one line: What is context window?"

    print("CPU-style request (num_gpu=0):")
    cpu_options = {
        "num_ctx": 1024,
        "num_gpu": 0,
        "num_thread": 4,
        "temperature": 0.1
    }
    print("Options:", cpu_options)
    cpu_resp = ollama_generate(prompt=test_prompt, options=cpu_options, stream=False)
    print("Response:", cpu_resp)

    print("\n" + "-" * 70)

    print("GPU-style request (num_gpu=1):")
    gpu_options = {
        "num_ctx": 1024,
        "num_gpu": 1,
        "num_thread": 4,
        "temperature": 0.1
    }
    print("Options:", gpu_options)
    gpu_resp = ollama_generate(prompt=test_prompt, options=gpu_options, stream=False)
    print("Response:", gpu_resp)

demo_cpu_gpu_calls()

print("\nTeaching note:")
print("In real Ollama usage, hardware decides actual CPU/GPU behavior.")

CPU-style request (num_gpu=0):
Options: {'num_ctx': 1024, 'num_gpu': 0, 'num_thread': 4, 'temperature': 0.1}
Response: Context window means how much text the model can read in one request.

----------------------------------------------------------------------
GPU-style request (num_gpu=1):
Options: {'num_ctx': 1024, 'num_gpu': 1, 'num_thread': 4, 'temperature': 0.1}
Response: Context window means how much text the model can read in one request.

Teaching note:
In real Ollama usage, hardware decides actual CPU/GPU behavior.


#### Question from Usha Kommari

# General Questions: Evaluation, Model Variance, and Vectorization
This section covers:
- Q1: How to validate answer accuracy when retrieval comes from multiple documents
- Q2: Why different models produce different answers, and whether RAG reduces hallucinations
- Q3: Types of vectorization and how vectors are stored

## Q1. How do we validate whether answers are accurate when retrieving from multiple documents?
Use layered validation instead of manual page-by-page checks:
1. Retrieval quality checks (did we fetch the right chunks?)
2. Groundedness checks (is the answer supported by evidence?)
3. Correctness checks against reference answers (for test set)
4. Citation checks (source, page/chunk, and quote coverage)

3 docs that have the same question - How are we going to validate?

1. Return the citation -> Source
compare it wil the ground truth that we already created -> Similarty score / Citation coverage / Show the retrived chunk

In [ ]:
# Q1 Demo: Simple multi-document RAG evaluation template
import re
from difflib import SequenceMatcher

def normalize_text(s):
    """Lowercase and remove extra spaces for fair comparison."""
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s

def score_answer_similarity(pred, ref):
    """Score from 0 to 1: higher means closer to reference answer."""
    return SequenceMatcher(None, normalize_text(pred), normalize_text(ref)).ratio()

def score_citation_coverage(pred, evidence_snippets):
    """Simple groundedness score: how many evidence snippets appear in answer."""
    pred_n = normalize_text(pred)
    hits = 0
    for ev in evidence_snippets:
        key_part = normalize_text(ev)[:60]
        if key_part and key_part in pred_n:
            hits += 1
    return hits / max(1, len(evidence_snippets))

eval_rows = [
    {
        "question": "What is the refund eligibility period?",
        "reference_answer": "Refunds are allowed within 30 days of purchase with invoice.",
        "predicted_answer": "Customers can claim refunds within 30 days if they have the invoice.",
        "evidence": [
            "Refunds are allowed within 30 days of purchase.",
            "Invoice is required for refund processing."
        ]
    },
    {
        "question": "Can trial users access premium analytics?",
        "reference_answer": "Trial users cannot access premium analytics dashboards.",
        "predicted_answer": "Yes, trial users can access premium analytics.",
        "evidence": [
            "Trial plan excludes premium analytics dashboards."
        ]
    }
]

print("Q1 Evaluation Results")
print("=" * 70)
for row in eval_rows:
    sim = score_answer_similarity(row["predicted_answer"], row["reference_answer"])
    cov = score_citation_coverage(row["predicted_answer"], row["evidence"] )

    print(f"Q: {row['question']}")
    print(f"- Similarity score: {sim:.2f}")
    print(f"- Citation coverage score: {cov:.2f}")

    verdict = "PASS" if sim >= 0.75 and cov >= 0.50 else "REVIEW"
    print(f"- Verdict: {verdict}\n")

Q1 Evaluation Results
Q: What is the refund eligibility period?
- Similarity score: 0.56
- Citation coverage score: 0.00
- Verdict: REVIEW

Q: Can trial users access premium analytics?
- Similarity score: 0.81
- Citation coverage score: 0.00
- Verdict: REVIEW



## Q2. Why do different models give different answers to the same prompt? <br>
Does RAG reduce hallucinations?
Different models vary in:
- Training data mix and quality
- Alignment and safety tuning
- Decoding behavior and temperature defaults
- Reasoning capability and context handling

> RAG usually reduces hallucinations by grounding answers in retrieved evidence,
but it does not guarantee zero hallucination. Quality depends on retrieval quality and prompting.

In [ ]:
# Q2 Add-on: Model variance controls (temperature and top_p)

def generation_settings_note(temperature, top_p):
    """Return a simple explanation for decoding settings."""
    stability = "more stable" if temperature <= 0.2 else "more diverse"
    nucleus = "tighter token set" if top_p <= 0.8 else "wider token set"
    return (
        f"temperature={temperature} -> {stability} outputs; "
        f"top_p={top_p} -> {nucleus}."
    )

examples = [
    (0.0, 0.8),
    (0.2, 0.9),
    (0.7, 0.95),
]

print("Decoding guidance for classroom demos:")
for t, p in examples:
    print("-", generation_settings_note(t, p))

print("\nPractical default for factual Q&A: temperature=0.0 to 0.2, top_p<=0.8")

Retrieval quality:
- pointwise retrival
- re-ranking techni using LLM 
- list wise retrieval

The LLM are probabalistc models

temperature = 0 -> reduce variations

RAG is more of showcasing the evideces from the document; 



In [ ]:
# Q2 Demo: Why grounded prompts help reduce hallucinations

def ask_grounded(question, context_chunks, temperature=0.0):
    """Answer only from provided context. If no evidence, abstain."""
    context = "\n\n".join(context_chunks)

    # We display temperature so students see it as a model setting knob.
    temp_note = f"(temperature={temperature})"

    # Very simple rule for classroom clarity.
    if "15 minutes" in context.lower() and "sla" in question.lower():
        return f"P1 incidents must receive an initial response within 15 minutes. (from context) {temp_note}"
    return f"I do not have enough evidence. {temp_note}"

question = "What is the SLA response time for P1 incidents?"

good_context = [
    "Policy doc: P1 incidents must receive initial response within 15 minutes.",
    "Escalation doc: P1 requires immediate on-call acknowledgment."
]

bad_context = [
    "Marketing doc: We value customer support.",
    "Roadmap doc: New analytics feature planned next quarter."
]

print("Grounded with relevant context:\n")
print(ask_grounded(question, good_context, temperature=0.0))

print("\n" + "=" * 70 + "\n")
print("Grounded with irrelevant context (should abstain):\n")
print(ask_grounded(question, bad_context, temperature=0.0))

Grounded with relevant context:

P1 incidents must receive an initial response within 15 minutes. (from context) (temperature=0.0)


Grounded with irrelevant context (should abstain):

I do not have enough evidence. (temperature=0.0)


1. logging files -> 
2. Push the code to git
3. Create a docker image
4. git conflicts
----------------------
1. you have a logic -> Use Ai as a coding assistant

## Q3. Different types of vectorization and how vectors are stored
Common vectorization types in LLM systems:
1. Dense vectors (embeddings): continuous float vectors, semantic similarity
2. Sparse vectors (e.g., TF-IDF/BM25 style): high-dimensional mostly zeros
3. Hybrid retrieval: combines dense + sparse scoring
4. Multi-vector retrieval: multiple vectors per document/chunk

> Storage options:
- In-memory arrays (small demos)
- Vector databases (FAISS, Chroma, Milvus, Weaviate, Pinecone, Qdrant)
- Metadata + IDs + document references are stored alongside vectors

In [ ]:
# Q3 Demo: Super simple vectorization + storage (word-count vectors)
import json
import math
from pathlib import Path

# Small fixed vocabulary so students can follow each vector position.
VOCAB = ["p1", "response", "minutes", "support", "security", "data"]

def simple_embed(text):
    """Count word occurrences for each vocab term."""
    t = text.lower()
    return [t.count(word) for word in VOCAB]

def cosine(a, b):
    """Cosine similarity between two small vectors."""
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(y * y for y in b))
    return dot / (na * nb + 1e-12)

docs = [
    {"id": "d1", "source": "policy.pdf", "page": 4, "text": "P1 incidents need response within 15 minutes."},
    {"id": "d2", "source": "pricing.pdf", "page": 2, "text": "Pro plan includes priority support."},
    {"id": "d3", "source": "security.pdf", "page": 8, "text": "Data is encrypted at rest and in transit."},
]

# Build a tiny in-memory vector store.
vector_store = []
for row in docs:
    vector_store.append({**row, "vector": simple_embed(row["text"])})

query = "What is the response time for P1 issues?"
qvec = simple_embed(query)

scored = []
for row in vector_store:
    score = cosine(qvec, row["vector"])
    scored.append((score, row))

scored.sort(key=lambda x: x[0], reverse=True)

print("Vocabulary order:", VOCAB)
print("Top matches:")
for score, row in scored[:2]:
    print(f"score={score:.3f} | id={row['id']} | src={row['source']}#page={row['page']}")
    print(" text:", row["text"])

# Save vectors to a JSON file to show simple persistence.
persist_path = Path("toy_vector_store.json")
persist_path.write_text(json.dumps(vector_store, indent=2), encoding="utf-8")
print(f"\nSaved toy vectors to {persist_path.resolve()}")

Vocabulary order: ['p1', 'response', 'minutes', 'support', 'security', 'data']
Top matches:
score=0.816 | id=d1 | src=policy.pdf#page=4
 text: P1 incidents need response within 15 minutes.
score=0.000 | id=d2 | src=pricing.pdf#page=2
 text: Pro plan includes priority support.

Saved toy vectors to D:\Mentoring\learwithsarvesh\12weekcrash Course\Doubt Clearring Session\toy_vector_store.json


#### Soma Sekhar

## Q4. How do I convert any of these apps into a web app or website?
We will take the **RAG Grounded Q&A app** (from Q2) and convert it into a web app using **Streamlit**.

Why Streamlit?
- Pure Python — no HTML/CSS/JavaScript required
- Works great for AI/LLM demos and internal tools
- Runs locally and can be deployed to cloud (Streamlit Community Cloud, HuggingFace Spaces, etc.)

Required install:
```bash
pip install streamlit requests
```

Run command:
```bash
streamlit run rag_webapp.py
```

In [ ]:
# Q4 Step 1: Write a very simple Streamlit web app file
from pathlib import Path

APP_CODE = '''
import streamlit as st

st.set_page_config(page_title="Simple RAG Demo", layout="centered")
st.title("Simple RAG Q&A Demo")
st.write("Paste context, ask a question, get a simple grounded-style answer.")

context_text = st.text_area("Context", height=160)
question_text = st.text_input("Question")

if st.button("Answer"):
    if not context_text.strip() or not question_text.strip():
        st.warning("Please fill both context and question.")
    else:
        if "15 minutes" in context_text.lower() and "sla" in question_text.lower():
            st.success("P1 incidents require a 15-minute initial response. (from context)")
        else:
            st.info("I do not have enough evidence from the provided context.")
'''

app_path = Path("rag_webapp.py")
app_path.write_text(APP_CODE.strip(), encoding="utf-8")
print(f"Web app written to: {app_path.resolve()}")
print("Next step: install streamlit if needed.")

Web app written to: D:\Mentoring\learwithsarvesh\12weekcrash Course\Doubt Clearring Session\rag_webapp.py
Next step: install streamlit if needed.


In [ ]:
# Q4 Step 2: Check whether Streamlit is installed (safe classroom check)
import importlib.util

if importlib.util.find_spec("streamlit") is None:
    print("Streamlit is not installed yet.")
    print("Run this command manually:")
    print("python -m pip install streamlit")
else:
    print("Streamlit is already installed.")

Streamlit is already installed.


In [ ]:
# Q4 Step 3: Show the launch command (no blocking run inside notebook)
from pathlib import Path

app_file = Path("rag_webapp.py")
if not app_file.exists():
    print("Run Step 1 first to create rag_webapp.py")
else:
    print("To launch the app, run this command in terminal:")
    print("streamlit run rag_webapp.py")
    print("Expected URL: http://localhost:8501")

To launch the app, run this command in terminal:
streamlit run rag_webapp.py
Expected URL: http://localhost:8501
